# BioShield ML Model Training
## Supervised Learning for Biometric Security Analysis

This notebook trains two supervised classification models:
1. **Spoof Detector**: Classifies biometric attempts as Genuine or Fake
2. **Anomaly Detector**: Classifies authentication patterns as Normal or Anomalous

Both models use supervised learning with labeled training data.

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Generate Synthetic Training Data

Since we don't have real-world data yet, we'll generate synthetic training data based on realistic scenarios.

In [ ]:
def generate_spoof_detection_data(n_samples=5000):
    """
    Generate synthetic data for spoof detection with enhanced biometric features.
    
    Features:
    - auth_start_ms: Authentication start timestamp (ms since epoch)
    - sensor_acquire_ms: Sensor acquisition start timestamp (ms)
    - detection_ms: Detection timestamp (finger/face detected) (ms)
    - success_failure_ms: Success/failure timestamp (ms)
    - sensor_latency: Time from auth start to sensor acquisition (ms)
    - detection_latency: Time from sensor acquisition to detection (ms)
    - completion_latency: Time from detection to completion (ms)
    - total_duration: Total authentication time (ms)
    - retry_count: Number of retry attempts
    - failure_reason: Encoded failure reason (0=success, 1=timeout, 2=no_match, 3=cancelled, 4=hardware_error)
    - cpu_load: CPU load during authentication (0.0-1.0)
    - thermal_state: Device thermal state (0=normal, 1=warm, 2=hot, 3=critical)
    - screen_state: Screen state during auth (0=off, 1=on, 2=locked)
    - entropy: Randomness/complexity of biometric data (0-1)
    - hasCrypto: Whether cryptographic operations are detected (0 or 1)
    - networkFlag: Whether network activity during auth (0 or 1)
    
    Label:
    - 0: Genuine biometric
    - 1: Fake/spoofed biometric
    """
    np.random.seed(42)
    
    # Base timestamp for simulation (current time)
    base_time = 1700000000000  # Nov 2023 in ms
    
    # Generate genuine biometric samples (50%)
    n_genuine = n_samples // 2
    
    # Genuine: Normal timing breakdown
    genuine_sensor_latency = np.random.normal(50, 15, n_genuine)  # 35-65ms to acquire sensor
    genuine_detection_latency = np.random.normal(200, 40, n_genuine)  # 160-240ms to detect
    genuine_completion_latency = np.random.normal(50, 10, n_genuine)  # 40-60ms to complete
    genuine_total_duration = genuine_sensor_latency + genuine_detection_latency + genuine_completion_latency
    
    # Construct timestamps
    genuine_auth_start = base_time + np.random.randint(0, 86400000, n_genuine)  # Random time within day
    genuine_sensor_acquire = genuine_auth_start + genuine_sensor_latency
    genuine_detection = genuine_sensor_acquire + genuine_detection_latency
    genuine_success_failure = genuine_detection + genuine_completion_latency
    
    # Genuine: Low retry, mostly success
    genuine_retry_count = np.random.choice([0, 1, 2], n_genuine, p=[0.85, 0.12, 0.03])
    genuine_failure_reason = np.random.choice([0, 1, 2], n_genuine, p=[0.95, 0.03, 0.02])  # 95% success
    
    # Genuine: Normal device states
    genuine_cpu_load = np.random.beta(2, 5, n_genuine) * 0.6  # Low to moderate CPU (0-0.6)
    genuine_thermal_state = np.random.choice([0, 1], n_genuine, p=[0.9, 0.1])  # Mostly normal/warm
    genuine_screen_state = np.random.choice([1, 2], n_genuine, p=[0.7, 0.3])  # Screen on or locked
    
    # Genuine: High entropy, crypto present, minimal network
    genuine_entropy = np.random.beta(8, 2, n_genuine)  # High entropy (0.7-0.9)
    genuine_crypto = np.random.choice([0, 1], n_genuine, p=[0.1, 0.9])  # 90% have crypto
    genuine_network = np.random.choice([0, 1], n_genuine, p=[0.95, 0.05])  # 5% network
    
    genuine_labels = np.zeros(n_genuine)
    
    # Generate fake/spoofed biometric samples (50%)
    n_fake = n_samples - n_genuine
    
    # Fake: Suspicious timing (too fast or inconsistent)
    fake_sensor_latency = np.random.normal(10, 5, n_fake)  # Very fast sensor acquisition (suspicious)
    fake_detection_latency = np.random.normal(30, 15, n_fake)  # Very fast detection (suspicious)
    fake_completion_latency = np.random.normal(20, 8, n_fake)  # Fast completion
    fake_total_duration = fake_sensor_latency + fake_detection_latency + fake_completion_latency
    
    # Construct timestamps
    fake_auth_start = base_time + np.random.randint(0, 86400000, n_fake)
    fake_sensor_acquire = fake_auth_start + fake_sensor_latency
    fake_detection = fake_sensor_acquire + fake_detection_latency
    fake_success_failure = fake_detection + fake_completion_latency
    
    # Fake: Higher retry, more failures
    fake_retry_count = np.random.choice([0, 1, 2, 3, 4], n_fake, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    fake_failure_reason = np.random.choice([0, 1, 2, 3, 4], n_fake, p=[0.4, 0.2, 0.2, 0.1, 0.1])  # More failures
    
    # Fake: Higher CPU (processing spoofed data), varied thermal, screen often off
    fake_cpu_load = np.random.beta(5, 2, n_fake) * 0.8 + 0.2  # High CPU (0.2-1.0)
    fake_thermal_state = np.random.choice([0, 1, 2], n_fake, p=[0.5, 0.3, 0.2])  # More heat
    fake_screen_state = np.random.choice([0, 1, 2], n_fake, p=[0.4, 0.3, 0.3])  # Screen often off (suspicious)
    
    # Fake: Low entropy, less crypto, more network
    fake_entropy = np.random.beta(2, 8, n_fake)  # Low entropy (0.1-0.3)
    fake_crypto = np.random.choice([0, 1], n_fake, p=[0.8, 0.2])  # 80% no crypto
    fake_network = np.random.choice([0, 1], n_fake, p=[0.3, 0.7])  # 70% network activity
    
    fake_labels = np.ones(n_fake)
    
    # Combine datasets
    auth_start_ms = np.concatenate([genuine_auth_start, fake_auth_start])
    sensor_acquire_ms = np.concatenate([genuine_sensor_acquire, fake_sensor_acquire])
    detection_ms = np.concatenate([genuine_detection, fake_detection])
    success_failure_ms = np.concatenate([genuine_success_failure, fake_success_failure])
    sensor_latency = np.concatenate([genuine_sensor_latency, fake_sensor_latency])
    detection_latency = np.concatenate([genuine_detection_latency, fake_detection_latency])
    completion_latency = np.concatenate([genuine_completion_latency, fake_completion_latency])
    total_duration = np.concatenate([genuine_total_duration, fake_total_duration])
    retry_count = np.concatenate([genuine_retry_count, fake_retry_count])
    failure_reason = np.concatenate([genuine_failure_reason, fake_failure_reason])
    cpu_load = np.concatenate([genuine_cpu_load, fake_cpu_load])
    thermal_state = np.concatenate([genuine_thermal_state, fake_thermal_state])
    screen_state = np.concatenate([genuine_screen_state, fake_screen_state])
    entropy = np.concatenate([genuine_entropy, fake_entropy])
    hasCrypto = np.concatenate([genuine_crypto, fake_crypto])
    networkFlag = np.concatenate([genuine_network, fake_network])
    labels = np.concatenate([genuine_labels, fake_labels])
    
    # Clip values to valid ranges
    sensor_latency = np.clip(sensor_latency, 1, 500)
    detection_latency = np.clip(detection_latency, 1, 1000)
    completion_latency = np.clip(completion_latency, 1, 500)
    total_duration = np.clip(total_duration, 10, 2000)
    retry_count = np.clip(retry_count, 0, 10)
    cpu_load = np.clip(cpu_load, 0.0, 1.0)
    thermal_state = np.clip(thermal_state, 0, 3)
    screen_state = np.clip(screen_state, 0, 2)
    entropy = np.clip(entropy, 0, 1)
    
    # Create DataFrame
    df = pd.DataFrame({
        'auth_start_ms': auth_start_ms,
        'sensor_acquire_ms': sensor_acquire_ms,
        'detection_ms': detection_ms,
        'success_failure_ms': success_failure_ms,
        'sensor_latency': sensor_latency,
        'detection_latency': detection_latency,
        'completion_latency': completion_latency,
        'total_duration': total_duration,
        'retry_count': retry_count,
        'failure_reason': failure_reason,
        'cpu_load': cpu_load,
        'thermal_state': thermal_state,
        'screen_state': screen_state,
        'entropy': entropy,
        'hasCrypto': hasCrypto,
        'networkFlag': networkFlag,
        'label': labels
    })
    
    # Shuffle
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return df

# Generate data
spoof_data = generate_spoof_detection_data(5000)
print("Spoof Detection Dataset (Enhanced Features):")
print(spoof_data.head())
print(f"\nDataset shape: {spoof_data.shape}")
print(f"\nClass distribution:\n{spoof_data['label'].value_counts()}")
print(f"\nFeature summary:")
print(spoof_data.describe())

In [ ]:
def generate_anomaly_detection_data(n_samples=5000):
    """
    Generate synthetic data for anomaly detection with enhanced biometric features.
    
    Features:
    - auth_start_ms: Authentication start timestamp (ms since epoch)
    - sensor_acquire_ms: Sensor acquisition start timestamp (ms)
    - detection_ms: Detection timestamp (finger/face detected) (ms)
    - success_failure_ms: Success/failure timestamp (ms)
    - sensor_latency: Time from auth start to sensor acquisition (ms)
    - detection_latency: Time from sensor acquisition to detection (ms)
    - completion_latency: Time from detection to completion (ms)
    - total_duration: Total authentication time (ms)
    - retry_count: Number of retry attempts
    - failure_reason: Encoded failure reason (0=success, 1=timeout, 2=no_match, 3=cancelled, 4=hardware_error)
    - cpu_load: CPU load during authentication (0.0-1.0)
    - thermal_state: Device thermal state (0=normal, 1=warm, 2=hot, 3=critical)
    - screen_state: Screen state during auth (0=off, 1=on, 2=locked)
    - entropy: Randomness/complexity of biometric data (0-1)
    - timeOfDay: Hour of authentication (0-23)
    - dayOfWeek: Day of week (1-7, 1=Monday)
    
    Label:
    - 0: Normal authentication
    - 1: Anomalous authentication
    """
    np.random.seed(42)
    
    # Base timestamp for simulation
    base_time = 1700000000000  # Nov 2023 in ms
    
    # Generate normal authentication samples (70%)
    n_normal = int(n_samples * 0.7)
    
    # Normal: Consistent timing patterns
    normal_sensor_latency = np.random.normal(50, 12, n_normal)
    normal_detection_latency = np.random.normal(220, 35, n_normal)
    normal_completion_latency = np.random.normal(45, 8, n_normal)
    normal_total_duration = normal_sensor_latency + normal_detection_latency + normal_completion_latency
    
    # Normal: Business hours (7am-11pm)
    normal_timeOfDay = np.concatenate([
        np.random.normal(9, 2, n_normal//2),  # Morning peak (7-11am)
        np.random.normal(18, 3, n_normal//2)  # Evening peak (3-11pm)
    ])
    normal_timeOfDay = np.clip(normal_timeOfDay, 7, 23)
    
    # Construct timestamps for normal samples
    normal_hour_offset = (normal_timeOfDay * 3600000).astype(int)
    normal_auth_start = base_time + normal_hour_offset + np.random.randint(0, 3600000, n_normal)
    normal_sensor_acquire = normal_auth_start + normal_sensor_latency
    normal_detection = normal_sensor_acquire + normal_detection_latency
    normal_success_failure = normal_detection + normal_completion_latency
    
    # Normal: Weekday heavy, low retry, mostly success
    normal_dayOfWeek = np.random.choice(range(1, 8), n_normal, p=[0.15, 0.15, 0.15, 0.15, 0.15, 0.13, 0.12])
    normal_retry_count = np.random.choice([0, 1], n_normal, p=[0.9, 0.1])
    normal_failure_reason = np.random.choice([0, 1, 2], n_normal, p=[0.95, 0.03, 0.02])
    
    # Normal: Moderate CPU, normal thermal, screen mostly on
    normal_cpu_load = np.random.beta(3, 4, n_normal) * 0.5  # Low-moderate (0-0.5)
    normal_thermal_state = np.random.choice([0, 1], n_normal, p=[0.85, 0.15])
    normal_screen_state = np.random.choice([1, 2], n_normal, p=[0.75, 0.25])
    
    # Normal: Moderate-high entropy
    normal_entropy = np.random.beta(6, 3, n_normal)
    
    normal_labels = np.zeros(n_normal)
    
    # Generate anomalous authentication samples (30%)
    n_anomaly = n_samples - n_normal
    
    # Anomaly: Inconsistent/suspicious timing
    anomaly_sensor_latency = np.concatenate([
        np.random.normal(5, 3, n_anomaly//3),     # Too fast
        np.random.normal(150, 30, n_anomaly//3),  # Too slow
        np.random.normal(50, 40, n_anomaly//3)    # Highly variable
    ])
    anomaly_detection_latency = np.concatenate([
        np.random.normal(20, 10, n_anomaly//3),   # Too fast
        np.random.normal(600, 100, n_anomaly//3), # Too slow
        np.random.normal(220, 150, n_anomaly//3)  # Highly variable
    ])
    anomaly_completion_latency = np.random.normal(40, 25, n_anomaly)
    anomaly_total_duration = anomaly_sensor_latency + anomaly_detection_latency + anomaly_completion_latency
    
    # Anomaly: Unusual hours (1am-5am - suspicious)
    anomaly_timeOfDay = np.random.uniform(1, 5, n_anomaly)
    
    # Construct timestamps for anomaly samples
    anomaly_hour_offset = (anomaly_timeOfDay * 3600000).astype(int)
    anomaly_auth_start = base_time + anomaly_hour_offset + np.random.randint(0, 3600000, n_anomaly)
    anomaly_sensor_acquire = anomaly_auth_start + anomaly_sensor_latency
    anomaly_detection = anomaly_sensor_acquire + anomaly_detection_latency
    anomaly_success_failure = anomaly_detection + anomaly_completion_latency
    
    # Anomaly: Random days, higher retry, more failures
    anomaly_dayOfWeek = np.random.choice(range(1, 8), n_anomaly)
    anomaly_retry_count = np.random.choice([0, 1, 2, 3, 4], n_anomaly, p=[0.3, 0.3, 0.2, 0.1, 0.1])
    anomaly_failure_reason = np.random.choice([0, 1, 2, 3, 4], n_anomaly, p=[0.5, 0.2, 0.15, 0.1, 0.05])
    
    # Anomaly: Higher CPU, varied thermal, screen often off
    anomaly_cpu_load = np.random.beta(5, 2, n_anomaly) * 0.7 + 0.3  # High (0.3-1.0)
    anomaly_thermal_state = np.random.choice([0, 1, 2, 3], n_anomaly, p=[0.4, 0.3, 0.2, 0.1])
    anomaly_screen_state = np.random.choice([0, 1, 2], n_anomaly, p=[0.5, 0.25, 0.25])
    
    # Anomaly: Low entropy
    anomaly_entropy = np.random.beta(2, 6, n_anomaly)
    
    anomaly_labels = np.ones(n_anomaly)
    
    # Combine datasets
    auth_start_ms = np.concatenate([normal_auth_start, anomaly_auth_start])
    sensor_acquire_ms = np.concatenate([normal_sensor_acquire, anomaly_sensor_acquire])
    detection_ms = np.concatenate([normal_detection, anomaly_detection])
    success_failure_ms = np.concatenate([normal_success_failure, anomaly_success_failure])
    sensor_latency = np.concatenate([normal_sensor_latency, anomaly_sensor_latency])
    detection_latency = np.concatenate([normal_detection_latency, anomaly_detection_latency])
    completion_latency = np.concatenate([normal_completion_latency, anomaly_completion_latency])
    total_duration = np.concatenate([normal_total_duration, anomaly_total_duration])
    retry_count = np.concatenate([normal_retry_count, anomaly_retry_count])
    failure_reason = np.concatenate([normal_failure_reason, anomaly_failure_reason])
    cpu_load = np.concatenate([normal_cpu_load, anomaly_cpu_load])
    thermal_state = np.concatenate([normal_thermal_state, anomaly_thermal_state])
    screen_state = np.concatenate([normal_screen_state, anomaly_screen_state])
    entropy = np.concatenate([normal_entropy, anomaly_entropy])
    timeOfDay = np.concatenate([normal_timeOfDay, anomaly_timeOfDay])
    dayOfWeek = np.concatenate([normal_dayOfWeek, anomaly_dayOfWeek])
    labels = np.concatenate([normal_labels, anomaly_labels])
    
    # Clip values to valid ranges
    sensor_latency = np.clip(sensor_latency, 1, 500)
    detection_latency = np.clip(detection_latency, 1, 1500)
    completion_latency = np.clip(completion_latency, 1, 500)
    total_duration = np.clip(total_duration, 10, 2500)
    retry_count = np.clip(retry_count, 0, 10)
    cpu_load = np.clip(cpu_load, 0.0, 1.0)
    thermal_state = np.clip(thermal_state, 0, 3)
    screen_state = np.clip(screen_state, 0, 2)
    entropy = np.clip(entropy, 0, 1)
    timeOfDay = np.clip(timeOfDay, 0, 23)
    dayOfWeek = np.clip(dayOfWeek, 1, 7)
    
    # Create DataFrame
    df = pd.DataFrame({
        'auth_start_ms': auth_start_ms,
        'sensor_acquire_ms': sensor_acquire_ms,
        'detection_ms': detection_ms,
        'success_failure_ms': success_failure_ms,
        'sensor_latency': sensor_latency,
        'detection_latency': detection_latency,
        'completion_latency': completion_latency,
        'total_duration': total_duration,
        'retry_count': retry_count,
        'failure_reason': failure_reason,
        'cpu_load': cpu_load,
        'thermal_state': thermal_state,
        'screen_state': screen_state,
        'entropy': entropy,
        'timeOfDay': timeOfDay,
        'dayOfWeek': dayOfWeek,
        'label': labels
    })
    
    # Shuffle
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return df

# Generate data
anomaly_data = generate_anomaly_detection_data(5000)
print("Anomaly Detection Dataset (Enhanced Features):")
print(anomaly_data.head())
print(f"\nDataset shape: {anomaly_data.shape}")
print(f"\nClass distribution:\n{anomaly_data['label'].value_counts()}")
print(f"\nFeature summary:")
print(anomaly_data.describe())

## 3. Data Visualization

In [ ]:
# Visualize spoof detection features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Duration
axes[0, 0].hist(spoof_data[spoof_data['label']==0]['duration'], bins=50, alpha=0.7, label='Genuine', color='green')
axes[0, 0].hist(spoof_data[spoof_data['label']==1]['duration'], bins=50, alpha=0.7, label='Fake', color='red')
axes[0, 0].set_xlabel('Duration (ms)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Spoof Detection: Duration Distribution')
axes[0, 0].legend()

# Entropy
axes[0, 1].hist(spoof_data[spoof_data['label']==0]['entropy'], bins=50, alpha=0.7, label='Genuine', color='green')
axes[0, 1].hist(spoof_data[spoof_data['label']==1]['entropy'], bins=50, alpha=0.7, label='Fake', color='red')
axes[0, 1].set_xlabel('Entropy')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Spoof Detection: Entropy Distribution')
axes[0, 1].legend()

# Has Crypto
crypto_counts = spoof_data.groupby(['label', 'hasCrypto']).size().unstack(fill_value=0)
crypto_counts.plot(kind='bar', ax=axes[1, 0], color=['red', 'green'])
axes[1, 0].set_xlabel('Label (0=Genuine, 1=Fake)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Spoof Detection: Crypto Usage')
axes[1, 0].set_xticklabels(['Genuine', 'Fake'], rotation=0)
axes[1, 0].legend(['No Crypto', 'Has Crypto'])

# Network Flag
network_counts = spoof_data.groupby(['label', 'networkFlag']).size().unstack(fill_value=0)
network_counts.plot(kind='bar', ax=axes[1, 1], color=['green', 'red'])
axes[1, 1].set_xlabel('Label (0=Genuine, 1=Fake)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Spoof Detection: Network Activity')
axes[1, 1].set_xticklabels(['Genuine', 'Fake'], rotation=0)
axes[1, 1].legend(['No Network', 'Network Active'])

plt.tight_layout()
plt.show()

In [ ]:
# Visualize anomaly detection features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Duration
axes[0, 0].hist(anomaly_data[anomaly_data['label']==0]['duration'], bins=50, alpha=0.7, label='Normal', color='blue')
axes[0, 0].hist(anomaly_data[anomaly_data['label']==1]['duration'], bins=50, alpha=0.7, label='Anomaly', color='orange')
axes[0, 0].set_xlabel('Duration (ms)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Anomaly Detection: Duration Distribution')
axes[0, 0].legend()

# Entropy
axes[0, 1].hist(anomaly_data[anomaly_data['label']==0]['entropy'], bins=50, alpha=0.7, label='Normal', color='blue')
axes[0, 1].hist(anomaly_data[anomaly_data['label']==1]['entropy'], bins=50, alpha=0.7, label='Anomaly', color='orange')
axes[0, 1].set_xlabel('Entropy')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Anomaly Detection: Entropy Distribution')
axes[0, 1].legend()

# Time of Day
axes[1, 0].hist(anomaly_data[anomaly_data['label']==0]['timeOfDay'], bins=24, alpha=0.7, label='Normal', color='blue')
axes[1, 0].hist(anomaly_data[anomaly_data['label']==1]['timeOfDay'], bins=24, alpha=0.7, label='Anomaly', color='orange')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Anomaly Detection: Time of Day')
axes[1, 0].legend()

# Day of Week
axes[1, 1].hist(anomaly_data[anomaly_data['label']==0]['dayOfWeek'], bins=7, alpha=0.7, label='Normal', color='blue', range=(1,8))
axes[1, 1].hist(anomaly_data[anomaly_data['label']==1]['dayOfWeek'], bins=7, alpha=0.7, label='Anomaly', color='orange', range=(1,8))
axes[1, 1].set_xlabel('Day of Week (1=Mon, 7=Sun)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Anomaly Detection: Day of Week')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Prepare Training Data

In [ ]:
def prepare_data(df, feature_columns):
    """
    Prepare data for training: split and normalize.
    """
    # Separate features and labels
    X = df[feature_columns].values
    y = df['label'].values
    
    # Split into train/validation/test (70/15/15)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    # Normalize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    print(f"Training set: {X_train.shape}")
    print(f"Validation set: {X_val.shape}")
    print(f"Test set: {X_test.shape}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

# Prepare spoof detection data with ENHANCED FEATURES
print("Preparing Spoof Detection data (Enhanced)...")
spoof_features = [
    'sensor_latency', 'detection_latency', 'completion_latency', 'total_duration',
    'retry_count', 'failure_reason', 'cpu_load', 'thermal_state', 'screen_state',
    'entropy', 'hasCrypto', 'networkFlag'
]
print(f"Spoof features ({len(spoof_features)}): {spoof_features}")
X_train_spoof, X_val_spoof, X_test_spoof, y_train_spoof, y_val_spoof, y_test_spoof, scaler_spoof = prepare_data(
    spoof_data, spoof_features
)

print("\nPreparing Anomaly Detection data (Enhanced)...")
anomaly_features = [
    'sensor_latency', 'detection_latency', 'completion_latency', 'total_duration',
    'retry_count', 'failure_reason', 'cpu_load', 'thermal_state', 'screen_state',
    'entropy', 'timeOfDay', 'dayOfWeek'
]
print(f"Anomaly features ({len(anomaly_features)}): {anomaly_features}")
X_train_anomaly, X_val_anomaly, X_test_anomaly, y_train_anomaly, y_val_anomaly, y_test_anomaly, scaler_anomaly = prepare_data(
    anomaly_data, anomaly_features
)

# Save scaler parameters for later use in Flutter
print("\n" + "="*80)
print("SCALER PARAMETERS (for Flutter MLModelService)")
print("="*80)
print("\nSpoof Detector Scaler:")
print(f"  Mean: {scaler_spoof.mean_.tolist()}")
print(f"  Std:  {scaler_spoof.scale_.tolist()}")
print("\nAnomaly Detector Scaler:")
print(f"  Mean: {scaler_anomaly.mean_.tolist()}")
print(f"  Std:  {scaler_anomaly.scale_.tolist()}")
print("="*80)

## 5. Build and Train Models

In [ ]:
def build_model(input_dim, name="model"):
    """
    Build a supervised binary classification model.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(64, activation='relu', name='dense1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', name='dense2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(16, activation='relu', name='dense3'),
        tf.keras.layers.Dense(1, activation='sigmoid', name='output')
    ], name=name)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

### 5.1 Train Spoof Detector

In [ ]:
print("Building Spoof Detector model with enhanced features...")
spoof_model = build_model(input_dim=12, name="spoof_detector")  # Updated from 4 to 12 features
spoof_model.summary()

In [ ]:
print("Training Spoof Detector...")

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

# Train
history_spoof = spoof_model.fit(
    X_train_spoof, y_train_spoof,
    validation_data=(X_val_spoof, y_val_spoof),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

### 5.2 Train Anomaly Detector

In [ ]:
print("Building Anomaly Detector model with enhanced features...")
anomaly_model = build_model(input_dim=12, name="anomaly_detector")  # Updated from 4 to 12 features
anomaly_model.summary()

In [ ]:
print("Training Anomaly Detector...")

# Train
history_anomaly = anomaly_model.fit(
    X_train_anomaly, y_train_anomaly,
    validation_data=(X_val_anomaly, y_val_anomaly),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 6. Evaluate Models

In [ ]:
def plot_training_history(history, title):
    """
    Plot training and validation metrics.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train Accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history_spoof, "Spoof Detector")
plot_training_history(history_anomaly, "Anomaly Detector")

In [ ]:
def evaluate_model(model, X_test, y_test, title):
    """
    Comprehensive model evaluation.
    """
    # Predictions
    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    
    # Metrics
    print(f"\n{'='*60}")
    print(f"{title} - Test Set Evaluation")
    print(f"{'='*60}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{title} - ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()
    
    return y_pred, y_pred_proba

# Evaluate Spoof Detector
y_pred_spoof, y_pred_proba_spoof = evaluate_model(
    spoof_model, X_test_spoof, y_test_spoof, "Spoof Detector"
)

# Evaluate Anomaly Detector
y_pred_anomaly, y_pred_proba_anomaly = evaluate_model(
    anomaly_model, X_test_anomaly, y_test_anomaly, "Anomaly Detector"
)

## 7. Convert to TensorFlow Lite

In [ ]:
def convert_to_tflite(model, model_name):
    """
    Convert Keras model to TensorFlow Lite format.
    """
    # Convert model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Optimization (optional - reduces model size)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Convert
    tflite_model = converter.convert()
    
    # Save
    tflite_filename = f'{model_name}.tflite'
    with open(tflite_filename, 'wb') as f:
        f.write(tflite_model)
    
    print(f"✅ Saved {tflite_filename} ({len(tflite_model)/1024:.2f} KB)")
    return tflite_filename

# Convert both models
spoof_tflite_file = convert_to_tflite(spoof_model, 'spoof_detector')
anomaly_tflite_file = convert_to_tflite(anomaly_model, 'anomaly_detector')

## 8. Test TFLite Models

In [ ]:
def test_tflite_model(tflite_filename, X_test, y_test):
    """
    Test TFLite model to ensure it works correctly.
    """
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_filename)
    interpreter.allocate_tensors()
    
    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"\nTesting {tflite_filename}")
    print(f"Input shape: {input_details[0]['shape']}")
    print(f"Output shape: {output_details[0]['shape']}")
    
    # Test on first 10 samples
    correct = 0
    for i in range(min(10, len(X_test))):
        # Prepare input
        input_data = np.array([X_test[i]], dtype=np.float32)
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        # Run inference
        interpreter.invoke()
        
        # Get output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        prediction = 1 if output_data[0][0] > 0.5 else 0
        
        print(f"Sample {i}: Pred={prediction}, Actual={int(y_test[i])}, Prob={output_data[0][0]:.3f}")
        
        if prediction == int(y_test[i]):
            correct += 1
    
    print(f"\nAccuracy on 10 samples: {correct}/10 = {correct/10*100:.1f}%")

# Test both TFLite models
test_tflite_model(spoof_tflite_file, X_test_spoof, y_test_spoof)
test_tflite_model(anomaly_tflite_file, X_test_anomaly, y_test_anomaly)

## 9. Download Models

In [ ]:
# Download the TFLite models
from google.colab import files

print("Downloading TFLite models...")
files.download('spoof_detector.tflite')
files.download('anomaly_detector.tflite')
print("\n✅ Models downloaded successfully!")
print("\nNext steps:")
print("1. Copy both .tflite files to: C:\\Users\\User\\Desktop\\School\\FYP\\BioShield\\assets\\models\\")
print("2. The ML service will automatically detect and load them")
print("3. Test your app to verify the models work correctly")

## 10. Model Information Summary

In [ ]:
print("="*80)
print("MODEL TRAINING COMPLETE - ENHANCED BIOMETRIC FEATURES")
print("="*80)

print("\n1. SPOOF DETECTOR")
print("   - Model Type: Supervised Binary Classification")
print("   - Input Features (12):")
print("     * Timing: sensor_latency, detection_latency, completion_latency, total_duration")
print("     * Failure Analysis: retry_count, failure_reason")
print("     * Device State: cpu_load, thermal_state, screen_state")
print("     * Biometric Quality: entropy, hasCrypto, networkFlag")
print("   - Output: Probability of being fake/spoofed (0-1)")
print("   - Threshold: 0.5 (>0.5 = fake, <=0.5 = genuine)")
print("   - File: spoof_detector.tflite")

print("\n2. ANOMALY DETECTOR")
print("   - Model Type: Supervised Binary Classification")
print("   - Input Features (12):")
print("     * Timing: sensor_latency, detection_latency, completion_latency, total_duration")
print("     * Failure Analysis: retry_count, failure_reason")
print("     * Device State: cpu_load, thermal_state, screen_state")
print("     * Biometric Quality: entropy")
print("     * Temporal Context: timeOfDay, dayOfWeek")
print("   - Output: Probability of being anomalous (0-1)")
print("   - Threshold: 0.5 (>0.5 = anomaly, <=0.5 = normal)")
print("   - File: anomaly_detector.tflite")

print("\n3. REQUIRED LOGGING (from Hooks/ContentProvider)")
print("   Hook logging MUST capture:")
print("   ✓ Authentication start timestamp (auth_start_ms)")
print("   ✓ Sensor acquisition start timestamp (sensor_acquire_ms)")
print("   ✓ Detection timestamp - finger/face detected (detection_ms)")
print("   ✓ Success/failure timestamp (success_failure_ms)")
print("   ✓ Retry count (retry_count)")
print("   ✓ Failure reason: 0=success, 1=timeout, 2=no_match, 3=cancelled, 4=hardware_error")
print("   ✓ CPU load during auth (cpu_load: 0.0-1.0)")
print("   ✓ Thermal state: 0=normal, 1=warm, 2=hot, 3=critical")
print("   ✓ Screen state: 0=off, 1=on, 2=locked")
print("   ✓ Entropy/complexity of biometric data")
print("   ✓ Crypto operations detected (hasCrypto: 0 or 1)")
print("   ✓ Network activity during auth (networkFlag: 0 or 1)")

print("\n4. FLUTTER INTEGRATION")
print("   - Service: lib/services/ml_model_service.dart")
print("   - The service MUST be updated to:")
print("     * Accept 12 input features (currently only accepts 4)")
print("     * Calculate derived features: sensor_latency, detection_latency, completion_latency")
print("     * Extract timeOfDay and dayOfWeek from timestamps")
print("     * Use updated scaler parameters (mean/std for 12 features)")
print("   - Place .tflite files in: assets/models/")

print("\n5. DATA NORMALIZATION")
print("   - Both models expect normalized inputs (StandardScaler)")
print("   - Features are scaled to have mean=0 and std=1")
print("   - MLModelService must use scaler parameters from training output above")

print("\n6. NEXT STEPS")
print("   1. Train this notebook to generate new .tflite models")
print("   2. Update vulnerable app hooks to capture all required metrics")
print("   3. Update BioShield MLModelService to process 12 features")
print("   4. Copy new .tflite models to assets/models/")
print("   5. Test end-to-end pipeline")

print("\n" + "="*80)